# Approach3 OPERA Hierarchy Acquisition Prototype

This notebook tests the first Python/geemap prototype for the Approach3 hierarchy: Dynamic World first, OPERA DSWx-HLS next, and OPERA DSWx-S1 last. It builds one acquisition-date product anchored on the first Dynamic World image in a post-OPERA test window.

In [ ]:
# Required parameters
EE_PROJECT = "your-google-cloud-project-id"

# Optional AOI/date parameters
HYBAS_ID = 1041259950
ANCHOR_START_DATE = "2025-01-01"
ANCHOR_END_DATE = "2025-02-01"  # Earth Engine filterDate end is exclusive.

# Optional Dynamic World thresholds, matching the JavaScript baseline defaults
DW_WATER_THRESHOLD = 0.5
DW_NONWATER_THRESHOLD = 0.05
DW_FLOODED_VEG_THRESHOLD = 0.3

# Optional pairing parameters
HLS_PAIR_WINDOW_DAYS = 1
S1_PAIR_WINDOW_DAYS = 3
INCLUDE_OPERA_HLS_SENTINEL2 = False

# Optional display and summary parameters
MAP_CENTER = [-6.5, 29.6]
MAP_ZOOM = 7
SUMMARY_SCALE_METERS = 300  # Use 30 for finer QA after the prototype is working.

## Imports And Earth Engine Initialization

Run JupyterLab from the repository root using the `Python (SW_DWS1)` kernel. Authenticate Earth Engine once from the project environment with `earthengine authenticate` if needed.

In [ ]:
from pathlib import Path
import sys

import ee
import geemap

current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    src_path = candidate / "Approaches" / "Approach3" / "src"
    if src_path.exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find Approach3/src from the current notebook path.")

sys.path.insert(0, str(REPO_ROOT / "Approaches" / "Approach3" / "src"))

from sw_dws1_approach3.datasets import DynamicWorldThresholds
from sw_dws1_approach3.gee_session import initialize_earth_engine
from sw_dws1_approach3.products import (
    PairingConfig,
    ProductConfig,
    build_acquisition_product,
    build_monthly_product,
    candidate_counts,
    first_dynamic_world_image,
    tanganyika_basin_aoi,
)

initialize_earth_engine(project=EE_PROJECT)
print("Earth Engine initialized")

## Build AOI And Check Source Availability

The default period is after OPERA DSWx-S1 availability begins, so all three source families should be testable if they intersect the AOI.

In [ ]:
aoi = tanganyika_basin_aoi(HYBAS_ID)

counts = candidate_counts(
    aoi=aoi,
    start_date=ANCHOR_START_DATE,
    end_date=ANCHOR_END_DATE,
    include_opera_hls_sentinel2=INCLUDE_OPERA_HLS_SENTINEL2,
).getInfo()

if counts["dynamic_world"] == 0:
    raise ValueError("No Dynamic World images were found for the selected AOI/date window.")

counts

## Build One Acquisition-Date Product

The output is anchored on the first Dynamic World image in the date window. OPERA-HLS and OPERA-S1 are paired by nearest acquisition time inside the configured windows. HLS Sentinel-2 is excluded by default.

In [ ]:
thresholds = DynamicWorldThresholds(
    water=DW_WATER_THRESHOLD,
    nonwater=DW_NONWATER_THRESHOLD,
    flooded_vegetation=DW_FLOODED_VEG_THRESHOLD,
)
pairing = PairingConfig(
    hls_pair_window_days=HLS_PAIR_WINDOW_DAYS,
    s1_pair_window_days=S1_PAIR_WINDOW_DAYS,
    include_opera_hls_sentinel2=INCLUDE_OPERA_HLS_SENTINEL2,
)
config = ProductConfig(thresholds=thresholds, pairing=pairing)

anchor_dw = first_dynamic_world_image(
    aoi=aoi,
    start_date=ANCHOR_START_DATE,
    end_date=ANCHOR_END_DATE,
)
product = build_acquisition_product(anchor_dw, aoi=aoi, config=config)

product_metadata = product.toDictionary(
    [
        "approach3_mode",
        "anchor_date",
        "hls_pair_window_days",
        "s1_pair_window_days",
        "include_opera_hls_sentinel2",
        "hls_candidate_count",
        "s1_candidate_count",
        "dynamic_world_source_id",
    ]
).getInfo()

product_metadata

## Visual QA In geemap

The source layer uses the class-specific source codes documented in `Approaches/Approach3/docs/method_design.md`.

In [ ]:
source_palette = [
    "000000",  # 0 no source
    "0066cc",  # 1 DW open water
    "33a02c",  # 2 DW flooded vegetation
    "6a3d9a",  # 3 DW both
    "1f78b4",  # 4 HLS open water
    "b2df8a",  # 5 HLS partial surface water
    "a6cee3",  # 6 S1 open water
    "fb9a99",  # 7 S1 inundated vegetation
    "d9d9d9",  # 8 DW non-water
    "bdbdbd",  # 9 HLS non-water
    "969696",  # 10 S1 non-water
]

m = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
m.add_basemap("HYBRID")
m.addLayer(aoi, {}, "HydroBASINS AOI")
m.addLayer(
    product.select("water"),
    {"min": 0, "max": 1, "palette": ["f0f0f0", "0055ff"]},
    "Approach3 water",
)
m.addLayer(
    product.select("water_class"),
    {"min": 0, "max": 3, "palette": ["f0f0f0", "0066cc", "33a02c", "6a3d9a"]},
    "Approach3 water_class",
)
m.addLayer(
    product.select("source"),
    {"min": 0, "max": 10, "palette": source_palette},
    "Approach3 source",
)
m.addLayer(
    product.select("source_rank"),
    {"min": 1, "max": 3, "palette": ["0066cc", "ff7f00", "e31a1c"]},
    "Approach3 source_rank",
)
m.centerObject(aoi, MAP_ZOOM)
m

## Compact Area QA

This uses the configurable summary scale above. It is intentionally coarse by default so the first notebook run is quick.

In [ ]:
area_km2 = ee.Image.pixelArea().divide(1_000_000).rename("area_km2")

source_area = area_km2.addBands(product.select("source")).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="source"),
    geometry=aoi,
    scale=SUMMARY_SCALE_METERS,
    maxPixels=1e13,
    bestEffort=True,
)
class_area = area_km2.addBands(product.select("water_class")).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="water_class"),
    geometry=aoi,
    scale=SUMMARY_SCALE_METERS,
    maxPixels=1e13,
    bestEffort=True,
)

{
    "source_area_km2": source_area.getInfo(),
    "water_class_area_km2": class_area.getInfo(),
}

## Monthly Aggregate Prototype

This monthly product aggregates each source family once inside the selected period, then applies the same strict hierarchy. Use `source_first_yyyymmdd` and `source_last_yyyymmdd` for monthly provenance.

In [ ]:
monthly_product = build_monthly_product(
    aoi=aoi,
    start_date=ANCHOR_START_DATE,
    end_date=ANCHOR_END_DATE,
    config=config,
)

monthly_metadata = monthly_product.toDictionary(
    [
        "approach3_mode",
        "period_start",
        "period_end",
        "monthly_reduce_method",
        "include_opera_hls_sentinel2",
        "dynamic_world_count",
        "hls_count",
        "s1_count",
    ]
).getInfo()

monthly_metadata

In [ ]:
m_monthly = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
m_monthly.add_basemap("HYBRID")
m_monthly.addLayer(aoi, {}, "HydroBASINS AOI")
m_monthly.addLayer(
    monthly_product.select("water"),
    {"min": 0, "max": 1, "palette": ["f0f0f0", "0055ff"]},
    "Monthly Approach3 water",
)
m_monthly.addLayer(
    monthly_product.select("water_class"),
    {"min": 0, "max": 3, "palette": ["f0f0f0", "0066cc", "33a02c", "6a3d9a"]},
    "Monthly Approach3 water_class",
)
m_monthly.addLayer(
    monthly_product.select("source"),
    {"min": 0, "max": 10, "palette": source_palette},
    "Monthly Approach3 source",
)
m_monthly.centerObject(aoi, MAP_ZOOM)
m_monthly

In [ ]:
monthly_source_area = area_km2.addBands(monthly_product.select("source")).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="source"),
    geometry=aoi,
    scale=SUMMARY_SCALE_METERS,
    maxPixels=1e13,
    bestEffort=True,
)
monthly_class_area = area_km2.addBands(monthly_product.select("water_class")).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="water_class"),
    geometry=aoi,
    scale=SUMMARY_SCALE_METERS,
    maxPixels=1e13,
    bestEffort=True,
)

{
    "monthly_source_area_km2": monthly_source_area.getInfo(),
    "monthly_water_class_area_km2": monthly_class_area.getInfo(),
}